# Isochrone Maps — Theory + Visualization

**Goal:** Understand how isochrones are computed (multi-source Dijkstra), 
then build a visualization showing how travel-time zones shrink and grow.

**Run time:** ~15 minutes

---

## What is an Isochrone?

An isochrone is a ** contour** of equal travel time. 
Every point inside the isochrone is reachable from the center within T minutes.

The word comes from Greek: *iso* = equal, *chronos* = time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import heapq

%matplotlib inline
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['font.size'] = 10

---

## 1. Multi-Source Dijkstra (Isochrone Computation)

A standard shortest-path query: find path from A to B.
An isochrone: find ALL points reachable from A within T minutes.

Same algorithm — just don't stop when you hit the goal, stop when `cost > max_time`.

In [ ]:
EMPTY, WALL = 0, 1

def make_grid():
    """25×25 grid with a corridor layout."""
    g = np.zeros((25, 25), dtype=int)
    g[0, :], g[-1, :], g[:, 0], g[:, -1] = 1, 1, 1, 1
    # Horizontal corridors
    g[5, 3:20] = 1
    g[12, 5:22] = 1
    g[19, 3:18] = 1
    # Vertical connectors
    g[5:13, 10] = 1
    g[12:20, 16] = 1
    return g

def get_neighbors(r, c, grid):
    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr, nc = r+dr, c+dc
        if 0 <= nr < grid.shape[0] and 0 <= nc < grid.shape[1]:
            if grid[nr, nc] != WALL:
                yield nr, nc

def compute_isochrone(grid, start, max_time):
    """
    Multi-source Dijkstra: expand from start until all
    reachable nodes have been visited.
    Returns dict: {node -> arrival_time}
    """
    dist = {start: 0}
    pq = [(0, start)]
    visited = set()
    
    while pq:
        d, (r, c) = heapq.heappop(pq)
        if (r, c) in visited:
            continue
        visited.add((r, c))
        
        if d > max_time:
            continue  # keep exploring for other short paths, don't add to result
        
        for nr, nc in get_neighbors(r, c, grid):
            new_dist = d + 1  # uniform cost
            if new_dist < dist.get((nr, nc), float('inf')):
                dist[(nr, nc)] = new_dist
                heapq.heappush(pq, (new_dist, (nr, nc)))
    
    return dist

# Test
START = (2, 2)
grid = make_grid()
grid[START[0], START[1]] = 0

reachable_5 = {k: v for k, v in compute_isochrone(grid, START, 5).items() if v <= 5}
reachable_10 = {k: v for k, v in compute_isochrone(grid, START, 10).items() if v <= 10}
reachable_15 = {k: v for k, v in compute_isochrone(grid, START, 15).items() if v <= 15}

print(f'Within 5 steps:  {len(reachable_5)} cells')
print(f'Within 10 steps: {len(reachable_10)} cells')
print(f'Within 15 steps: {len(reachable_15)} cells')

---

## 2. Visualizing Isochrones as Contours

In [ ]:
def draw_isochrone_grid(grid, reachable, title, max_time):
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Background
    bg = np.zeros_like(grid, dtype=float)
    bg[grid == WALL] = -1  # walls = -1
    
    # Fill reachable cells with arrival time
    for (r, c), t in reachable.items():
        bg[r, c] = t
    
    # Draw contours
    levels = [0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 7.5, 10.5, 15.5]
    contour_colors = ['#3fb950','#58a6ff','#d29922','#f85149','#bc8cff']
    
    # Plot filled contours
    masked = np.ma.masked_where(bg < 0, bg)
    cmap = plt.cm.YlOrRd
    im = ax.imshow(masked, cmap=cmap, vmin=0, vmax=max_time, interpolation='nearest')
    
    # Overlay contour lines
    contour_levels = list(range(1, max_time + 1))
    cs = ax.contour(bg, levels=contour_levels, colors='white', linewidths=0.8, alpha=0.5)
    ax.clabel(cs, inline=True, fontsize=7, fmt='%d')
    
    # Mark start
    ax.add_patch(plt.Circle((START[1], START[0]), 0.5, color='#3fb950', zorder=5))
    ax.text(START[1], START[0], 'S', ha='center', va='center', fontsize=8, fontweight='bold', color='white', zorder=6)
    
    # Walls
    for r in range(grid.shape[0]):
        for c in range(grid.shape[1]):
            if grid[r, c] == WALL:
                ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, color='#21262d', zorder=3))
    
    ax.set_title(title, fontsize=14)
    ax.axis('off')
    plt.colorbar(im, ax=ax, label='Travel time (steps)', shrink=0.7)
    plt.tight_layout()
    plt.show()

draw_isochrone_grid(grid, reachable_15, 'Isochrone Map — All points reachable within N steps from S', 15)

---

## 3. How Isochrones Deform Around Walls

In an open grid, isochrones are perfect circles.
Obstacles deform them — the isochrone "bends" around walls.

In [ ]:
# Compare open grid vs grid with a wall

open_grid = np.zeros((20, 20), dtype=int)
open_grid[0, :], open_grid[-1, :], open_grid[:, 0], open_grid[:, -1] = 1, 1, 1, 1

wall_grid = open_grid.copy()
wall_grid[5:16, 10] = 1  # vertical wall splitting the grid

start = (10, 2)

open_iso = {k: v for k, v in compute_isochrone(open_grid, start, 8).items() if v <= 8}
wall_iso = {k: v for k, v in compute_isochrone(wall_grid, start, 8).items() if v <= 8}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

for ax, iso_grid, iso, label in [(ax1, open_grid, open_iso, 'Open Grid'),
                                   (ax2, wall_grid, wall_iso, 'Wall Splits Grid')]:
    bg = np.zeros_like(iso_grid, dtype=float)
    bg[iso_grid == WALL] = -1
    for (r, c), t in iso.items():
        bg[r, c] = t
    masked = np.ma.masked_where(bg < 0, bg)
    ax.imshow(masked, cmap='YlOrRd', vmin=0, vmax=8, interpolation='nearest')
    cs = ax.contour(bg, levels=list(range(1,9)), colors='white', linewidths=0.8, alpha=0.6)
    ax.clabel(cs, inline=True, fontsize=7, fmt='%d')
    ax.add_patch(plt.Circle((start[1], start[0]), 0.5, color='#3fb950', zorder=5))
    ax.set_title(f'{label} — {len(iso)} cells reachable in 8 steps', fontsize=12)
    ax.axis('off')

plt.suptitle('Same 8-step Radius — How Walls Deform the Isochrone', fontsize=14)
plt.tight_layout()
plt.show()

---

## 4. The Rush Hour Effect (Simulated)

In BAS 5 Minute, different travel modes change speed. 
Walking: ~5 km/h. Cycling: ~15 km/h. Driving: ~40 km/h (no traffic).

We can simulate this by multiplying the grid cost.

In [ ]:
def compute_isochrone_weighted(grid, start, max_time, speed_factor):
    """
    Higher speed_factor = cheaper edges = larger isochrone.
    In real routing: highway has speed_limit=100 km/h, residential=30 km/h.
    Here we simulate by scaling edge costs.
    """
    dist = {start: 0}
    pq = [(0, start)]
    visited = set()
    
    while pq:
        d, (r, c) = heapq.heappop(pq)
        if (r, c) in visited:
            continue
        visited.add((r, c))
        if d > max_time:
            continue
        for nr, nc in get_neighbors(r, c, grid):
            new_dist = d + speed_factor  # weighted edge cost
            if new_dist < dist.get((nr, nc), float('inf')):
                dist[(nr, nc)] = new_dist
                heapq.heappush(pq, (new_dist, (nr, nc)))
    
    return {k: v for k, v in dist.items() if v <= max_time}

# Simulate walking (slow), cycling, driving
start = (2, 2)
walking  = compute_isochrone_weighted(grid, start, 15, speed_factor=3.0)   # slow
cycling  = compute_isochrone_weighted(grid, start, 15, speed_factor=1.0)   # medium
driving  = compute_isochrone_weighted(grid, start, 15, speed_factor=0.4)  # fast

print(f'Walking simulation:  {len(walking)} cells')
print(f'Cycling simulation:  {len(cycling)} cells')
print(f'Driving simulation:  {len(driving)} cells')

---

## Key Takeaways

1. **Isochrone = all points reachable within T minutes** — computed with multi-source Dijkstra
2. **Walls deform the contour** — isochrones aren't circles in the real world
3. **Speed matters quadratically** — double the speed = 4x the area reachable
4. **This is what OpenRouteService does** — but on a 50M-edge road network instead of a 25×25 grid